# 05 · Conformal volatility management of factor portfolios: H3 (design Sections 7 and 8.3)

Exposure set at each month end and held for the next month: w = min(2, c_t / V̂), with c_t set in real time so that
the average past exposure is 1. Strategies: unmanaged, VM-RV (Moreira-Muir), VM-HAR, VM-DL, **CVM** (squared
conformal upper bound on DLinear), CVM-HAR. H3 is a secondary hypothesis. Inverse-variance rule (primary) and inverse-volatility rule.
Costs 10 bp per unit change in exposure; cost sensitivity 0-50 bp.

The CVM coverage level is chosen once on the development period (pooled Sharpe) and frozen; the holdout reports
all four levels as sensitivity. **Primary test:** pooled panel, CVM vs VM-DL, Ledoit-Wolf (2011) variance test.
Per-factor results use Romano-Wolf step-down p-values. Sharpe ratios are descriptive.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
VM = {}
for name in pp.FACTOR_PANELS:
    VM[name] = pp.stage_vm(cfg, name)
    print("done", name, "| CVM level:", pp.load_selections(cfg)["cvm_level"].get(name))

In [ ]:
for name in pp.FACTOR_PANELS:
    display(Markdown(f"## {name}"))
    S = VM[name]["summary"]
    if not len(S):
        print("no live months in this period"); continue
    display(Markdown("**Pooled factors, inverse-variance rule**"))
    display(S[(S["asset"] == "POOLED") & (S["rule"] == "var")].round(4))
    display(Markdown("**H3 tests (pooled = primary)**"))
    T3 = VM[name]["H3"]
    display(T3[T3["asset"] == "POOLED"].round(4))
    show(f"fig_{name}_vm_wealth"); show(f"fig_{name}_vm_exposure")
    display(Markdown("**Per factor (Romano-Wolf adjusted)**"))
    display(T3[(T3["asset"] != "POOLED") & (T3["rule"] == "var")][["asset", "comparator", "vol_ratio", "p_var_lower",
            "p_var_romano_wolf", "mdd_diff", "sharpe_cvm", "sharpe_comp"]].round(4))
    display(Markdown("**CVM level sensitivity and cost sensitivity**"))
    table(f"{name}_cvm_level_sensitivity"); table(f"{name}_vm_cost_sensitivity")
    display(Markdown("**Moreira-Muir alphas**")); table(f"{name}_mm_alpha")